# 🎨 Qwen-Image-2.1: Local GPU Inference in Google Colab

### ⚡ Zero API Endpoints • 100% Local GPU Execution • Stored Permanently in Google Drive

This notebook downloads the complete model weights for **Qwen/Qwen-Image-2.1** (~25 GB) **directly into your Google Drive** (`MyDrive/AI_Models/Qwen-Image-2.1/`) using Google Colab's high-speed cloud connection.

* **No local PC disk space used**: Everything transfers between Hugging Face and Google Drive in the cloud.
* **No Hugging Face API limits**: You are not calling an API endpoint; the model runs 100% locally on your Colab GPU.
* **One-Time Download**: Once saved in your Google Drive (256 GB space), future Colab sessions load it instantly from Drive without re-downloading!

### Step 1: Check GPU Runtime
Ensure your notebook is connected to a GPU (`Runtime` ➔ `Change runtime type` ➔ `T4 GPU` or `A100/L4`).

In [ ]:
!nvidia-smi

### Step 2: Install Libraries
Install the latest `torchao`, `diffusers`, `transformers`, `huggingface_hub`, and `accelerate`.

In [ ]:
# Upgrade torchao and install required libraries
!pip install -q -U torchao
!pip install -q git+https://github.com/huggingface/diffusers.git
!pip install -q transformers>=4.48.0 accelerate>=1.2.0 safetensors sentencepiece huggingface_hub gradio bitsandbytes

### Step 3: Mount Your Google Drive (256 GB Space)
Connect your Google Drive so the model weights and generated images are permanently stored in your account.

In [ ]:
import os
from google.colab import drive

# Mount Drive
drive.mount('/content/drive')

# Define folder paths in Google Drive
MODEL_DRIVE_PATH = "/content/drive/MyDrive/AI_Models/Qwen-Image-2.1"
OUTPUT_IMAGES_DIR = "/content/drive/MyDrive/AI_Models/Generated_Images"

os.makedirs(MODEL_DRIVE_PATH, exist_ok=True)
os.makedirs(OUTPUT_IMAGES_DIR, exist_ok=True)

print(f"📁 Model Directory: {MODEL_DRIVE_PATH}")
print(f"🖼️ Image Output Directory: {OUTPUT_IMAGES_DIR}")

### Step 4: Download Model Weights Directly to Google Drive (One-Time)
This runs directly on Colab's high-speed datacenter connection and writes files straight into your Google Drive.

> **Note:** If you have already downloaded the model previously, it will automatically detect existing files and skip re-downloading.

In [ ]:
import os
from huggingface_hub import snapshot_download

REPO_ID = "Qwen/Qwen-Image-2.1"

# Check if model files already exist in Google Drive
existing_files = os.listdir(MODEL_DRIVE_PATH) if os.path.exists(MODEL_DRIVE_PATH) else []
weights_present = any(f.endswith('.safetensors') or f.endswith('.bin') for f in existing_files)

if weights_present:
    print(f"✅ Model weights already found in {MODEL_DRIVE_PATH}. Skipping download!")
else:
    print(f"🚀 Downloading {REPO_ID} directly to your Google Drive...")
    print("This may take a few minutes depending on network speed, but uses 0 bytes on your local PC!")
    
    # Optional HuggingFace token if required (leave None for public downloads)
    HF_TOKEN = None  # e.g., "hf_yourTokenHere" if needed
    
    downloaded_dir = snapshot_download(
        repo_id=REPO_ID,
        local_dir=MODEL_DRIVE_PATH,
        local_dir_use_symlinks=False,
        token=HF_TOKEN,
        resume_download=True,
        max_workers=4
    )
    print(f"🎉 Download completed! Weights stored permanently at: {downloaded_dir}")

### Step 5: Load Model in 4-bit Directly onto GPU (Zero CPU RAM Crash)
By quantizing to 4-bit (NF4) and loading directly to GPU with `device_map="cuda"`, the entire model fits inside ~7.8 GB of GPU VRAM, completely bypassing Colab's 12.7 GB CPU RAM limit.

In [ ]:
import torch
from diffusers import QwenImage21Pipeline
from diffusers.quantizers import PipelineQuantizationConfig

print(f"Loading model weights locally from: {MODEL_DRIVE_PATH} in 4-bit (NF4)...")

# 4-bit NormalFloat (NF4) quantization configuration
quant_config = PipelineQuantizationConfig(
    quant_backend="bitsandbytes_4bit",
    quant_kwargs={
        "load_in_4bit": True,
        "bnb_4bit_quant_type": "nf4",
        "bnb_4bit_compute_dtype": torch.float16,
        "bnb_4bit_use_double_quant": True,
    },
    components_to_quantize=["transformer", "text_encoder"],
)

# Load pipeline directly into GPU memory (prevents System CPU RAM crash)
pipe = QwenImage21Pipeline.from_pretrained(
    MODEL_DRIVE_PATH,
    quantization_config=quant_config,
    torch_dtype=torch.float16,
    device_map="cuda",
    low_cpu_mem_usage=True,
    local_files_only=True
)

# Enable VAE tiling to eliminate memory spikes during image decoding
pipe.vae.enable_tiling()

print("✨ Model successfully loaded in 4-bit directly on GPU! Ready for inference!")

### Step 6: Quick Test Run (Direct GPU Generation)
Test generation of a single image and save it directly to Google Drive.

In [ ]:
import gc
import time
from datetime import datetime

# Clean up any lingering system RAM & GPU memory
gc.collect()
torch.cuda.empty_cache()

prompt = "A hyper-detailed cinematic portrait of a robotic samurai in a bamboo forest, golden hour lighting, photorealistic, 8k"

print(f"Running local GPU inference for prompt:\n\"{prompt}\"\n")
start_time = time.time()

image = pipe(
    prompt=prompt,
    num_inference_steps=30,
    width=1024,
    height=1024,
).images[0]

elapsed = time.time() - start_time
print(f"Done in {elapsed:.2f} seconds!")

# Save directly to Google Drive
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
save_path = f"{OUTPUT_IMAGES_DIR}/qwen_test_{timestamp}.png"
image.save(save_path)
print(f"💾 Image saved to Google Drive: {save_path}")

# Display result
display(image)

### Step 7: Launch Interactive Web Studio (Gradio)
Run an interactive web interface with sliders, prompts, and a public URL to generate unlimited images.

In [ ]:
import gradio as gr
from datetime import datetime
import torch
from PIL import Image

# ==========================================
# ENDPOINT 1: Multi-Reference Image Generation
# Passes Empty Template + 4 Product Images directly into Qwen 2.1
# ==========================================
def generate_catalogue_multi(
    template_img,
    prod1,
    prod2,
    prod3,
    prod4,
    prompt,
    neg_prompt,
    steps,
    guidance_scale,
    seed
):
    if template_img is None:
        return None, "Error: Template image is required."
        
    ref_images = [template_img]
    for p in [prod1, prod2, prod3, prod4]:
        if p is not None:
            ref_images.append(p)
            
    generator = None
    if seed != -1:
        generator = torch.Generator(device="cuda").manual_seed(int(seed))
        
    try:
        output = pipe(
            prompt=prompt,
            negative_prompt=neg_prompt if neg_prompt and neg_prompt.strip() else None,
            image=ref_images if len(ref_images) > 1 else ref_images[0],
            num_inference_steps=int(steps),
            true_cfg_scale=float(guidance_scale),
            generator=generator
        ).images[0]
    except TypeError:
        output = pipe(
            prompt=prompt,
            negative_prompt=neg_prompt if neg_prompt and neg_prompt.strip() else None,
            image=ref_images if len(ref_images) > 1 else ref_images[0],
            num_inference_steps=int(steps),
            guidance_scale=float(guidance_scale),
            generator=generator
        ).images[0]
        
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    file_path = f"{OUTPUT_IMAGES_DIR}/catalogue_multi_{timestamp}.png"
    output.save(file_path)
    return output, f"Saved to Google Drive: {file_path}"


# ==========================================
# ENDPOINT 2: Fast Relighting & Harmonization
# Passes pre-composited layout into Qwen for studio reflections and lighting
# ==========================================
def relight_composite(
    composite_img,
    prompt,
    neg_prompt,
    steps,
    guidance_scale,
    seed
):
    if composite_img is None:
        return None, "Error: Composite image is required."
        
    generator = None
    if seed != -1:
        generator = torch.Generator(device="cuda").manual_seed(int(seed))
        
    try:
        output = pipe(
            prompt=prompt,
            negative_prompt=neg_prompt if neg_prompt and neg_prompt.strip() else None,
            image=composite_img,
            num_inference_steps=int(steps),
            true_cfg_scale=float(guidance_scale),
            generator=generator
        ).images[0]
    except TypeError:
        output = pipe(
            prompt=prompt,
            negative_prompt=neg_prompt if neg_prompt and neg_prompt.strip() else None,
            image=composite_img,
            num_inference_steps=int(steps),
            guidance_scale=float(guidance_scale),
            generator=generator
        ).images[0]
        
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    file_path = f"{OUTPUT_IMAGES_DIR}/relight_{timestamp}.png"
    output.save(file_path)
    return output, f"Saved to Google Drive: {file_path}"


# ==========================================
# ENDPOINT 3: Original Text-to-Image (Preserved)
# ==========================================
def generate_image_local(prompt, neg_prompt, steps, width, height, guidance_scale, seed):
    generator = None
    if seed != -1:
        generator = torch.Generator(device="cuda").manual_seed(int(seed))
        
    image = pipe(
        prompt=prompt,
        negative_prompt=neg_prompt if neg_prompt.strip() else None,
        num_inference_steps=int(steps),
        true_cfg_scale=float(guidance_scale),
        width=int(width),
        height=int(height),
        generator=generator,
    ).images[0]
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    file_path = f"{OUTPUT_IMAGES_DIR}/qwen_{timestamp}.png"
    image.save(file_path)
    
    return image, f"Saved to Google Drive: {file_path}"


# ==========================================
# Gradio Studio Interface with 3 Named Endpoints
# ==========================================
with gr.Blocks(title="Qwen-Image-2.1 Digital Catalogue Studio") as demo:
    gr.Markdown("# Qwen-Image-2.1 Digital Catalogue Studio")
    gr.Markdown("**Multi-Endpoint GPU Inference** • Multi-Image Inpainting • Fast Relighting • Text-to-Image")
    
    with gr.Tab("Multi-Product Catalogue Inpainting"):
        gr.Markdown("Pass empty template and up to 4 product images to place them naturally.")
        with gr.Row():
            with gr.Column():
                m_template = gr.Image(label="Empty Template", type="pil")
                with gr.Row():
                    m_p1 = gr.Image(label="Product 1 (Top-Left)", type="pil")
                    m_p2 = gr.Image(label="Product 2 (Top-Right)", type="pil")
                with gr.Row():
                    m_p3 = gr.Image(label="Product 3 (Bottom-Left)", type="pil")
                    m_p4 = gr.Image(label="Product 4 (Bottom-Right)", type="pil")
                m_prompt = gr.Textbox(
                    label="Prompt",
                    value="Place these product images inside template such that they look professionally placed and natural on the marble pedestals. Realistic studio lighting, reflections. Do not name those products or add text."
                )
                m_neg = gr.Textbox(label="Negative Prompt", value="blurry, distorted, low quality, warped, watermark, extra text, product names")
                with gr.Row():
                    m_steps = gr.Slider(10, 50, value=30, step=1, label="Steps")
                    m_cfg = gr.Slider(1.0, 10.0, value=1.0, step=0.5, label="Guidance Scale")
                m_seed = gr.Number(value=-1, label="Seed (-1 for random)")
                m_btn = gr.Button("Generate Multi-Product Catalogue", variant="primary")
            with gr.Column():
                m_out = gr.Image(label="Catalogue Output", type="pil")
                m_status = gr.Textbox(label="Status / Save Path", interactive=False)
                
        m_btn.click(
            fn=generate_catalogue_multi,
            inputs=[m_template, m_p1, m_p2, m_p3, m_p4, m_prompt, m_neg, m_steps, m_cfg, m_seed],
            outputs=[m_out, m_status],
            api_name="generate_catalogue_multi"
        )
        
    with gr.Tab("Fast Studio Relighting and Reflections"):
        gr.Markdown("Send a pre-composited layout for AI studio lighting, contact shadows, and marble reflections.")
        with gr.Row():
            with gr.Column():
                r_comp = gr.Image(label="Composite Image", type="pil")
                r_prompt = gr.Textbox(
                    label="Prompt",
                    value="High-end luxury studio commercial photography, warm golden reflections on polished marble, photorealistic contact shadows, 8k product render. Keep all ribbons blank."
                )
                r_neg = gr.Textbox(label="Negative Prompt", value="blurry, distorted, low quality, text on ribbons, logos")
                with gr.Row():
                    r_steps = gr.Slider(10, 40, value=20, step=1, label="Steps (Fast)")
                    r_cfg = gr.Slider(1.0, 10.0, value=1.0, step=0.5, label="Guidance Scale")
                r_seed = gr.Number(value=-1, label="Seed (-1 for random)")
                r_btn = gr.Button("Relight and Harmonize", variant="primary")
            with gr.Column():
                r_out = gr.Image(label="Relit Catalogue", type="pil")
                r_status = gr.Textbox(label="Status / Save Path", interactive=False)
                
        r_btn.click(
            fn=relight_composite,
            inputs=[r_comp, r_prompt, r_neg, r_steps, r_cfg, r_seed],
            outputs=[r_out, r_status],
            api_name="relight_composite"
        )

    with gr.Tab("Text-to-Image (Original)"):
        with gr.Row():
            with gr.Column():
                t_prompt = gr.Textbox(label="Prompt", lines=3)
                t_neg = gr.Textbox(label="Negative Prompt", lines=2)
                with gr.Row():
                    t_w = gr.Slider(512, 2048, step=64, value=1024, label="Width")
                    t_h = gr.Slider(512, 2048, step=64, value=1024, label="Height")
                with gr.Row():
                    t_steps = gr.Slider(10, 50, step=1, value=30, label="Steps")
                    t_cfg = gr.Slider(1.0, 10.0, step=0.5, value=1.0, label="Guidance Scale")
                t_seed = gr.Number(value=-1, label="Seed")
                t_btn = gr.Button("Generate Text-to-Image", variant="primary")
            with gr.Column():
                t_out = gr.Image(label="Generated Image", type="pil")
                t_status = gr.Textbox(label="Save Location", interactive=False)
                
        t_btn.click(
            fn=generate_image_local,
            inputs=[t_prompt, t_neg, t_steps, t_w, t_h, t_cfg, t_seed],
            outputs=[t_out, t_status],
            api_name="generate_image_local"
        )

demo.queue(max_size=20).launch(share=True, debug=True)
